# 03. Entrenamiento y registro con MLflow

**Proyecto**: Pronóstico de demanda multi-series (Curso II — Especialización ML Engineering)

Este notebook replica el flujo de Fase 3:

1. **Preprocesamiento**: subconjunto de **150 series** (10 tiendas × 15 artículos,
   mezcla de demanda alta/media/baja) para que el dataset quede < 100 MB, con
   features construidas sobre la **serie completa antes del split** (sin fuga y
   con holdout completo de 90 días).
2. **Entrenamiento**: baselines (naive, seasonal-naive, media) y LightGBM, cada
   uno registrado como run de MLflow + Model Registry.
3. **Selección**: el alias `Production` apunta al mejor modelo según la regla de
   decisión (ranking MASE + WAPE con filtro de sesgo relativo <= 5%).

Todo queda en la base local `mlruns/mlflow.db`. Para explorarla:
`python -m mlflow ui` (o abrir http://127.0.0.1:5000 si el servidor ya corre).

> Reutiliza los módulos de `src/` para que la lógica esté en un solo lugar:
> `src/features/build_features.py`, `src/models/metrics.py` y `src/models/train_model.py`.


In [1]:
%matplotlib inline
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import mlflow
from mlflow.tracking import MlflowClient

# Localiza la raíz del proyecto estés donde estés (desde notebooks/ o desde la raíz)
def _find_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / "data" / "raw" / "train.csv").exists():
            return p
    raise FileNotFoundError("No se encontró la raíz del proyecto (data/raw/train.csv)")

ROOT = _find_root(Path.cwd())
sys.path.insert(0, str(ROOT))

from src.features.build_features import build_features, FEATURE_COLUMNS  # noqa: E402
from src.models.metrics import naive_scale_by_series  # noqa: E402
from src.models.train_model import (  # noqa: E402
    MAX_REL_BIAS_PCT,
    MeanRegressor,
    ModelConfig,
    NaiveRegressor,
    SeasonalNaiveRegressor,
    promote_best_model,
    train_and_log_model,
)

sns.set_theme()
pd.set_option("display.float_format", "{:.2f}".format)

DATA_RAW = ROOT / "data" / "raw" / "train.csv"
DATA_OUT = ROOT / "data" / "processed"
FIGURES = ROOT / "reports" / "figures"
FIGURES.mkdir(parents=True, exist_ok=True)

TRACKING_URI = f"sqlite:///{(ROOT / 'mlruns' / 'mlflow.db').as_posix()}"
EXPERIMENT_NAME = "demand_forecast_fase3"
REGISTERED_MODEL = "demand_forecast"
TARGET = "sales"
TRAIN_CUTOFF = "2017-09-30"

mlflow.set_tracking_uri(TRACKING_URI)
mlflow.set_experiment(EXPERIMENT_NAME)
print("OK")


E:\JAIME\2023 3000\Estadística\18 Gemini CLI\ML2_Series_de_tiempo\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


OK


## 1. Preprocesamiento: subconjunto de 150 series

El dataset crudo tiene 913k filas (500 series), demasiado para el límite de 100 MB
de MLflow con artefactos. Se conservan 150 series (10 tiendas × 15 artículos),
elegidas para mezclar demanda alta, media y baja:

```python
ITEMS = [1, 2, 5, 6, 7, 8, 13, 14, 15, 16, 23, 24, 25, 28, 49]
STORES = list(range(1, 11))
```

Las features (calendario, lags y rolling) se construyen sobre la **serie completa**
y recién después se divide por fecha (`train <= 2017-09-30`, `holdout = oct-dic 2017`).
Así el holdout conserva los 90 días completos con lags/variables móviles válidos,
usando siempre solo historial pasado (sin fuga de datos).


In [2]:
DEFAULT_ITEMS = [1, 2, 5, 6, 7, 8, 13, 14, 15, 16, 23, 24, 25, 28, 49]
DEFAULT_STORES = list(range(1, 11))

raw = pd.read_csv(DATA_RAW, parse_dates=["date"])
full = raw[
    raw["store"].isin(DEFAULT_STORES) & raw["item"].isin(DEFAULT_ITEMS)
].sort_values(["store", "item", "date"]).reset_index(drop=True)

print(f"Filas originales: {len(raw):,} | Subconjunto: {len(full):,}")
print(f"Series: {full['store'].nunique()} tiendas x {full['item'].nunique()} items")
full.head()


Filas originales: 913,000 | Subconjunto: 273,900
Series: 10 tiendas x 15 items


,date,store,item,sales
0,2013-01-01,1,1,13
1,2013-01-02,1,1,11
2,2013-01-03,1,1,14
3,2013-01-04,1,1,13
4,2013-01-05,1,1,10


In [3]:
# Features sobre la serie completa, y luego split temporal
featured = build_features(full, target=TARGET)

train_feat = featured[featured["date"] <= TRAIN_CUTOFF].dropna().reset_index(drop=True)
holdout_feat = featured[featured["date"] > TRAIN_CUTOFF].copy().reset_index(drop=True)
if holdout_feat.isna().any().any():
    holdout_feat = holdout_feat.dropna().reset_index(drop=True)

train_raw = full[full["date"] <= TRAIN_CUTOFF].copy()
holdout_raw = full[full["date"] > TRAIN_CUTOFF].copy()

for df, name in [(train_raw, "train.csv"), (holdout_raw, "holdout.csv"),
                 (train_feat, "train_features.csv"), (holdout_feat, "holdout_features.csv")]:
    df.to_csv(DATA_OUT / name, index=False)
    print(f"{name:24s} {len(df):,} filas  {(DATA_OUT / name).stat().st_size / 1e6:6.2f} MB")

print(f"\nHoldout: {holdout_raw['date'].min().date()} .. {holdout_raw['date'].max().date()}"
      f" ({holdout_raw['date'].nunique()} días)")


train.csv                260,100 filas    5.15 MB
holdout.csv              13,800 filas    0.27 MB


train_features.csv       255,600 filas   30.91 MB
holdout_features.csv     13,800 filas    1.69 MB

Holdout: 2017-10-01 .. 2017-12-31 (92 días)


In [4]:
# Serie de ejemplo: ventas reales con el corte de train/holdout
example = full[full["item"] == 1]
fig, ax = plt.subplots(figsize=(10, 3))
ax.plot(example["date"], example["sales"], lw=0.8)
ax.axvline(pd.Timestamp(TRAIN_CUTOFF), color="red", ls="--", label="Split train/holdout")
ax.set_title("Ventas diarias — store 1, item 1")
ax.set_ylabel("Unidades")
ax.legend(); ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig(FIGURES / "serie_ejemplo_fase3.png", dpi=150)
plt.show()


C:\Users\Jaime\AppData\Local\Temp\ipykernel_18412\2826637320.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 2. Carga de datos para entrenar

Se cargan las features ya guardadas, la escala naive por serie (denominador del
MASE, calculada sobre el historial real de entrenamiento) y el `df_test` con
`(date, store, item, sales)` para poder cruzar las predicciones con lo real.


In [5]:
def load_data():
    train = pd.read_csv(DATA_OUT / "train_features.csv")
    holdout = pd.read_csv(DATA_OUT / "holdout_features.csv")
    train = train.dropna().reset_index(drop=True)
    holdout = holdout.dropna().reset_index(drop=True)

    features = [c for c in FEATURE_COLUMNS if c in train.columns]
    X_train, y_train = train[features], train[TARGET]
    X_test, y_test = holdout[features], holdout[TARGET]

    df_test = holdout[["date", "store", "item", "sales"]].reset_index(drop=True)
    series_ids_test = pd.Series(list(zip(df_test["store"], df_test["item"])))

    train_raw = pd.read_csv(DATA_OUT / "train.csv")
    naive_scale = naive_scale_by_series(train_raw)
    return X_train, X_test, y_train, y_test, naive_scale, df_test, series_ids_test

X_train, X_test, y_train, y_test, naive_scale, df_test, series_ids_test = load_data()
print(f"Train: {X_train.shape} | Holdout: {X_test.shape}")
print(f"Series en holdout: {series_ids_test.nunique()}")


Train: (255600, 15) | Holdout: (13800, 15)
Series en holdout: 150


## 3. Entrenamiento y registro en MLflow

Se definen las configuraciones: los tres baselines y LightGBM (parámetros fijos
para que la comparación sea limpia). Cada modelo se entrena, se evalúa sobre el
holdout y `train_and_log_model` registra:

- parámetros y todas las métricas (incl. `bias` y `rel_bias_pct`),
- artefactos: `predictions.csv` y la gráfica del pronóstico agregado,
- el modelo en el Model Registry (`demand_forecast`) con su versión.

### Regla de decisión (sesgo)

- `bias = mean(pred - real)`; `bias < 0` ⇒ el modelo **subestima** (riesgo de
  rotura de stock, invisible para RMSE/MAE).
- `rel_bias_pct = bias / mean(real) * 100`.
- Candidatos a `Production`: `|rel_bias_pct| <= 5%`; se ordenan por rango
  combinado MASE + WAPE y se desempata por menor `|bias|`.


In [6]:
BASE_MODELS = {
    "naive": ModelConfig(
        name="naive", family="baseline",
        estimator_factory=lambda: NaiveRegressor(),
        params={"type": "naive", "lag": 1},
    ),
    "seasonal_naive": ModelConfig(
        name="seasonal_naive", family="baseline",
        estimator_factory=lambda: SeasonalNaiveRegressor(),
        params={"type": "seasonal_naive", "lag": 7},
    ),
    "mean": ModelConfig(
        name="mean", family="baseline",
        estimator_factory=lambda: MeanRegressor(),
        params={"type": "mean", "window": 30},
    ),
}

from lightgbm import LGBMRegressor  # noqa: E402

lgb_params = {
    "n_estimators": 300, "learning_rate": 0.05, "num_leaves": 63,
    "random_state": 42, "n_jobs": -1, "verbose": -1,
}

MODELS = {
    **BASE_MODELS,
    "lightgbm": ModelConfig(
        name="lightgbm", family="gbm",
        estimator_factory=lambda: LGBMRegressor(**lgb_params),
        params=dict(lgb_params),
    ),
}

print("Modelos:", ", ".join(MODELS))
print(f"Filtro de sesgo: |rel_bias| <= {MAX_REL_BIAS_PCT}% para Production")


Modelos: naive, seasonal_naive, mean, lightgbm
Filtro de sesgo: |rel_bias| <= 5.0% para Production


In [7]:
# Entrena cada modelo y lo registra en MLflow (crea un run y una versión nueva)
results = {}
for name, cfg in MODELS.items():
    results[name] = train_and_log_model(
        cfg,
        X_train=X_train, y_train=y_train, X_test=X_test, y_test=y_test,
        df_test=df_test, series_ids_test=series_ids_test, naive_scale=naive_scale,
        experiment_name=EXPERIMENT_NAME,
        registered_model_name=REGISTERED_MODEL,
        feature_names=list(X_train.columns),
    )


Registered model 'demand_forecast' already exists. Creating a new version of this model...
2026/08/09 20:26:09 WARNING mlflow.tracking._model_registry.fluent: Run with id d81a48a73373486993cee1f9982c2df9 has no artifacts at artifact path 'model', registering model based on models:/m-ea4ea90dcc71464c8643a751b6c09f49 instead


Created version '6' of model 'demand_forecast'.


  -> Production actualizado: lightgbm (v5) MASE=0.581 WAPE=10.41% rel_bias=-0.25%
[naive] MASE=1.038 | WAPE=19.23% | bias=+0.18 | rel_bias=+0.31% | run=d81a48a7 | v6


Registered model 'demand_forecast' already exists. Creating a new version of this model...
2026/08/09 20:26:18 WARNING mlflow.tracking._model_registry.fluent: Run with id 28fc245448584060bd3eab912ab0efc7 has no artifacts at artifact path 'model', registering model based on models:/m-86a21eec62034c749a27c5dbcc73de16 instead


Created version '7' of model 'demand_forecast'.


...

Created version '8' of model 'demand_forecast'.


  -> Production actualizado: lightgbm (v5) MASE=0.581 WAPE=10.41% rel_bias=-0.25%
[mean] MASE=0.918 | WAPE=17.33% | bias=+3.13 | rel_bias=+5.32% | run=a42c40a8 | v8


Registered model 'demand_forecast' already exists. Creating a new version of this model...
2026/08/09 20:26:38 WARNING mlflow.tracking._model_registry.fluent: Run with id c535a6e6c6884d4db086a6e9e73f9eea has no artifacts at artifact path 'model', registering model based on models:/m-0dfbe4d096d04fa7993407b8893756a5 instead


  -> Production actualizado: lightgbm (v9) MASE=0.581 WAPE=10.41% rel_bias=-0.25%
[lightgbm] MASE=0.581 | WAPE=10.41% | bias=-0.15 | rel_bias=-0.25% | run=c535a6e6 | v9


Created version '9' of model 'demand_forecast'.


In [8]:
promote_best_model(EXPERIMENT_NAME, REGISTERED_MODEL, alias="Production")


  -> Production actualizado: lightgbm (v9) MASE=0.581 WAPE=10.41% rel_bias=-0.25%


True

In [9]:
# Tabla comparativa desde los runs de MLflow
def comparison_table():
    client = MlflowClient()
    exp = client.get_experiment_by_name(EXPERIMENT_NAME)
    rows = []
    for r in client.search_runs([exp.experiment_id]):
        m, p, t = r.data.metrics, r.data.params, r.data.tags
        version = p.get("registered_model_version")
        if version is None or "mase" not in m or "wape" not in m:
            continue
        try:
            mv = client.get_model_version(REGISTERED_MODEL, int(version))
        except Exception:
            continue
        rows.append({
            "modelo": t.get("mlflow.runName", r.info.run_id[:8]),
            "version": int(version),
            "MASE": m["mase"], "WAPE": m["wape"],
            "bias": m.get("bias", float("nan")),
            "rel_bias%": m.get("rel_bias_pct", float("nan")),
            "Production": "Production" in (mv.aliases or []),
        })
    return pd.DataFrame(rows).sort_values("MASE").reset_index(drop=True)

comparison_table().round(3)


,modelo,version,MASE,WAPE,bias,rel_bias%,Production
0,lightgbm,9,0.58,10.41,-0.15,-0.25,True
1,lightgbm,5,0.58,10.41,-0.15,-0.25,False
2,seasonal_naive,3,0.88,16.01,1.44,2.44,False
3,seasonal_naive,7,0.88,16.01,1.44,2.44,False
4,mean,4,0.92,17.33,3.13,5.32,False
5,mean,8,0.92,17.33,3.13,5.32,False
6,naive,6,1.04,19.23,0.18,0.31,False
7,naive,2,1.04,19.23,0.18,0.31,False


## 4. Modelo de producción

El alias `Production` del Model Registry apunta a la mejor versión según la regla
de decisión. Se carga con `models:/demand_forecast@Production` (sin acordarse de
qué modelo fue) y se genera una predicción de ejemplo sobre el holdout.


In [10]:
model = mlflow.pyfunc.load_model(f"models:/{REGISTERED_MODEL}@Production")
pred = model.predict(X_test.head(14))
df_test.head(14).assign(pred=pred)[["date", "store", "item", "sales", "pred"]].round(1)


,date,store,item,sales,pred
0,2017-10-01,1,1,21,27.10
1,2017-10-02,1,1,12,17.70
2,2017-10-03,1,1,18,20.20
3,2017-10-04,1,1,15,20.30
4,2017-10-05,1,1,20,21.60
5,2017-10-06,1,1,19,22.50
6,2017-10-07,1,1,22,24.20
7,2017-10-08,1,1,19,24.40
8,2017-10-09,1,1,9,16.00
9,2017-10-10,1,1,23,18.20


In [11]:
# Pronóstico agregado diario (real vs Production) sobre todo el holdout
y_pred_all = model.predict(X_test)
agg = df_test.assign(pred=y_pred_all).groupby("date")[["sales", "pred"]].sum()

fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(agg.index, agg["sales"], label="Real", marker="o", ms=3, lw=1.2)
ax.plot(agg.index, agg["pred"], label="Predicción (Production)", marker="x", ms=3, lw=1.2)
ax.set_title("Pronóstico agregado diario — holdout (oct-dic 2017)")
ax.set_xlabel("Fecha")
ax.set_ylabel("Unidades")
ax.legend(); ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig(FIGURES / "forecast_production.png", dpi=150)
plt.show()


C:\Users\Jaime\AppData\Local\Temp\ipykernel_18412\1084729891.py:14: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Conclusiones

- El pipeline completo (preprocesado → entrenamiento → registro → promoción) quedó
  reproducible en notebook reutilizando `src/`, igual que los scripts `preprocess.py`
  y `train.py`.
- El filtro de sesgo descarta modelos que subestiman/sobreestiman aunque tengan buen
  error puntual (caso típico: la media tiene buen MASE/WAPE pero sesgo > 5%).
- El modelo de producción se consume vía `models:/demand_forecast@Production`, sin
  depender de qué modelo ganó.
